# 階段 1：LLM 基本功（步驟 1 到 3）

對應文件：`docs/01-stage-llm-basics.md`。這份 notebook 帶你做步驟 1 到 3 的實驗；步驟 4、5（FastAPI、測試題）寫在 `src/` 與 `evals/`，照文件做。

**開始前**先照 [README「怎麼開始」](https://github.com/8loser/vuln-agent#怎麼開始) 設好環境。下一格會從 `src/vuln_agent/llm.py` 載入 `ask()`、`text()`、`cost()`。

每格都會呼叫 API、會花錢，**請一格一格跑，少用 Run All**。repo 裡的 notebook 不附執行結果，每次跑的結果都可能不同；程式格後面的「觀察」是維護者某一次執行的結果，拿來對照你自己的輸出。

標 `TODO` 的地方是要你自己完成的練習。

In [ ]:
# 改了 src/vuln_agent/*.py 之後不用重啟 kernel
%load_ext autoreload
%autoreload 2

from pathlib import Path

ROOT = Path.cwd().parent  # kernel 的工作目錄是 notebooks/，往上一層才是專案根目錄

from vuln_agent.llm import ask, client, cost, text, DEFAULT_MODEL

## 步驟 1：環境與第一次呼叫

直接用官方 SDK。`client.messages.create()` 回傳 SDK 的 `Message` 物件（Pydantic model），用屬性取值，例如 `resp.usage.input_tokens`。欄位定義在 SDK 原始碼 `anthropic/types/message.py`：

| 欄位 | 內容 |
|---|---|
| `id` | 這則回應的 ID |
| `type`、`role` | 固定是 `"message"`、`"assistant"` |
| `model` | 處理這次請求的模型 |
| `content` | 一串 block，每個 block 用 `type` 分辨形狀：`text`（文字在 `.text`）、`thinking`、`tool_use` 等；`llm.text()` 只取文字 |
| `stop_reason` | 為什麼停下來：`end_turn` 正常結束、`max_tokens` 被截斷、`tool_use` 要呼叫工具、`refusal` 被拒絕、`stop_sequence` 碰到自訂停止字串、`pause_turn` 長任務暫停、`model_context_window_exceeded` 超過 context window |
| `stop_sequence` | 碰到的自訂停止字串，沒有就是 `None` |
| `usage` | 計費依據：`input_tokens`、`output_tokens`；`output_tokens_details.thinking_tokens` 是輸出裡 thinking 占的部分；`cache_*` 屬於 prompt caching（階段 6） |

`resp._request_id` 是這次請求的 ID，回報問題時附上。它雖然以底線開頭，SDK 文件說明它是公開屬性。

### 1.1 第一次呼叫與回應結構

先呼叫一次，確認環境設好、API 能正常回答。

回答不是直接放在 `resp` 上：`resp.content` 是一串 block，文字放在 `type` 為 `"text"` 的 block 的 `.text` 裡。`text()`（見 `src/vuln_agent/llm.py`）把所有 text block 的 `.text` 接起來，略過 `thinking`、`tool_use` 等其他 block，所以不用管 text block 排在第幾個。

In [ ]:
resp = ask([{"role": "user", "content": "CVSS 和 EPSS 差在哪？"}], max_tokens=2048)
print(text(resp))

拆開同一個 `resp`：比較 Python 型別、`to_json()` 轉回的 API 原始 JSON，對照上表每個欄位。

In [ ]:
print(type(resp).__name__, [type(b).__name__ for b in resp.content], type(resp.usage).__name__)
print(resp.to_json())  # 只列 API 有回傳的欄位，key 跟 API 原始 JSON 一致
print("request id:", resp._request_id)

**觀察**：

- `content` 有兩個 block，先 `ThinkingBlock` 再 `TextBlock`：沒設定 thinking 參數，回應裡仍有 thinking block，所以 `resp.content[0].text` 會出錯，要用 `text()` 取
- thinking block 的 `thinking` 是空字串，只有 `signature`
- `output_tokens` 1341 裡 `thinking_tokens` 占 394：看不到的 thinking 也算進輸出計費
- `to_json()` 還列出表格沒提的欄位，例如 `container`、`diagnostics`、`stop_details`、`usage.inference_geo`、`usage.service_tier`，沒用到的功能值是 `null` 或 0

`max_tokens` 是這次回應輸出 token 的上限，thinking 和回答共用同一個上限（[官方文件](https://platform.claude.com/docs/en/build-with-claude/thinking-steering-and-cost)）。這格測試回答超過上限時會發生什麼事，以及程式怎麼知道回答被截斷了：同一題改成 `max_tokens=20`，跟上面正常結束的 `resp` 比較 `stop_reason`、`usage.output_tokens`、`content` 裡有哪些 block、`thinking_tokens` 和印出的回答。

In [ ]:
short = ask([{"role": "user", "content": "CVSS 和 EPSS 差在哪？"}], max_tokens=20)
print("stop_reason:", short.stop_reason, "output:", short.usage.output_tokens,
      "thinking:", short.usage.output_tokens_details.thinking_tokens)
print("content:", [b.type for b in short.content])
print(text(short))

**觀察**：

- `stop_reason` 是 `max_tokens`，`output_tokens` 剛好 20：上限用完就停，回答在字中間被切斷
- 這次 `thinking_tokens` 是 0，`content` 只有一個 text block；上面那格同一題有 thinking block，用了 394 個 thinking token
- text block 開頭是 `<reasoning>` 加一段簡體中文的推理，不是給使用者的回答

### 1.2 中英文 token 比較

先看什麼算一個 token。下一格用 `count_tokens` 只計算、不生成，量幾個片段各占幾個 token：英文常見字、長字與它的複數、中文單字與詞、數字、CVE 編號、漏洞俗稱。

`count_tokens` 回傳整則訊息的 token 數，除了送出的文字，還包含訊息格式本身的 token。這裡把「格式加上最少內容」的 token 數稱為**基本量**：先量一則只有 `.` 的訊息，結果就是基本量。訊息內容要有至少一個非空白字元，空字串或只有空白會被 API 拒絕（400 錯誤），所以用一個 `.` 當最少內容。每個片段的結果扣掉基本量，得到「比一個 `.` 多幾個 token」，片段之間可以互相比較。

In [ ]:
def count(s):
    return client.messages.count_tokens(model=DEFAULT_MODEL, messages=[{"role": "user", "content": s}]).input_tokens

base = count(".")
print("基本量:", base)
for s in ["a", "the", "vulnerability", "vulnerabilities", "漏", "漏洞", "弱點分析", "2021", "CVE-2021-44228", "log4shell"]:
    print(f"{s!r:20} {len(s):3} 字元  多 {count(s) - base} 個 token")

再拿下面要送出的完整問句，中英文各算一次整句，再拆開一段一段算、加總，比較整句和拆開加總的 token 數。拆法中英文用同一條規則：連續的英文字母與數字算一段（例如 `CVSS`、`difference`、`3`），其他每個字元各算一段（中文字、標點），空白不算。

這裡每一段都要扣掉格式的 token，但基本量裡還包含 `.` 自己的 token，直接扣基本量會讓每段少算一個 `.`，段數越多少算越多。所以這格扣 `base - 1`，也就是假設 `.` 占 1 個 token：上一格 `a`、`the` 扣完都是 0，跟 `.` 一樣多，而非空白的內容至少占 1 個 token。API 只回傳整則訊息的數量，這個假設沒辦法直接量出來驗證。

In [ ]:
import re

q_zh = "CVSS 和 EPSS 差在哪？用 3 句話回答。"
q_en = "What is the difference between CVSS and EPSS? Answer in 3 sentences."
fmt = base - 1  # 只扣格式，假設 `.` 占 1 個 token
for name, q in [("中文", q_zh), ("英文", q_en)]:
    parts = re.findall(r"[A-Za-z0-9]+|\S", q)
    each = [count(p) - fmt for p in parts]
    print(name, "整句", count(q) - fmt, "個 token，拆成", len(parts), "段，加總", sum(each), "個 token")
    print("  ", " ".join(f"{p}:{n}" for p, n in zip(parts, each)))

同一個問題中英文各送一次。注意 `output_tokens` 包含 thinking，要扣掉 `thinking_tokens` 才是回答本身。兩次回答的內容與長度都不同，比較時要記得這一點。

In [ ]:
zh = ask([{"role": "user", "content": "CVSS 和 EPSS 差在哪？用 3 句話回答。"}])
en = ask([{"role": "user", "content": "What is the difference between CVSS and EPSS? Answer in 3 sentences."}])
for name, r in [("中文", zh), ("英文", en)]:
    print(name, "input:", r.usage.input_tokens, "output:", r.usage.output_tokens,
          "thinking:", r.usage.output_tokens_details.thinking_tokens, "字元數:", len(text(r)))

再用 `count_tokens` 比較同一份回答的中英文版本，一樣扣掉基本量。

In [ ]:
# 只算 token，不生成回答：同一份回答的中英文版本
zh = ("CVSS 依漏洞的特性給出 0 到 10 分，衡量漏洞被利用後的嚴重程度。"
      "EPSS 由 FIRST 維護，預測漏洞在未來 30 天內被實際利用的機率，範圍是 0 到 1。"
      "排修補順序時兩者一起看：CVSS 說明影響多大，EPSS 說明多可能發生。")
en = ("CVSS scores a vulnerability from 0 to 10 based on its characteristics, measuring how severe it would be if exploited. "
      "EPSS, maintained by FIRST, predicts the probability that a vulnerability will be exploited in the wild within the next 30 days, on a scale from 0 to 1. "
      "When prioritizing fixes, use both: CVSS tells you how big the impact is, and EPSS tells you how likely it is to happen.")

for name, s in [("中文", zh), ("英文", en)]:
    n = count(s) - base
    print(name, n, "tokens", len(s), "字元", f"每 token {len(s) / n:.2f} 字元")

**觀察**：

- 中文約每個字 1 個多 token，英文約 2 到 3 個字母 1 個 token
- 成本主要看輸出長度與 thinking，而不是語言

### 1.3 穩定度與 effort

同一題跑 3 次，比較三次回答的措辭與內容。再拿一題需要推理的題目，用 `effort` low 和 high 各跑一次，比較 `output_tokens`、`thinking_tokens`、耗時與結論。參數以[官方文件](https://platform.claude.com/docs/en/api/messages/create)為準。

In [ ]:
import time

q = [{"role": "user", "content": "用一句話說明 EPSS。"}]
for _ in range(3):
    print(text(ask(q, max_tokens=200)))

hard = [{"role": "user", "content": "有 3 個漏洞：A 的 CVSS 9.8 但 EPSS 0.01；B 的 CVSS 7.5 且在 KEV 裡；C 的 CVSS 6.1 但 EPSS 0.6。只能先修一個，該修哪個？說明理由。"}]
for effort in ("low", "high"):
    t0 = time.time()
    r = ask(hard, max_tokens=4096, output_config={"effort": effort})
    print(f"--- effort={effort}", "output:", r.usage.output_tokens,
          "thinking:", r.usage.output_tokens_details.thinking_tokens, f"{time.time() - t0:.1f}s")
    print(text(r))

**觀察**：

- 同一題每次回答的措辭都不同，內容不保證一致
- effort 調高會多花 thinking token 與時間
- effort 調高，結論不一定跟著改變

### 1.4 換小模型比成本

**TODO**：先補完 `cost()`：改 `src/vuln_agent/llm.py`，改完存檔；`%autoreload` 會自動載入新版，不用重啟 kernel。依[官方定價](https://platform.claude.com/docs/en/about-claude/pricing)建一張「模型 → 每百萬 token 輸入／輸出價格」的表，把 `usage.input_tokens`、`usage.output_tokens` 換算成美元。進階：把 `cache_read_input_tokens`、`cache_creation_input_tokens` 也算進去（階段 6 會用到）。

下一格先用假的 usage 檢查你的 `cost()`，再實際比較兩個模型。

In [ ]:
from types import SimpleNamespace

fake = SimpleNamespace(input_tokens=1_000_000, output_tokens=1_000_000,
                       cache_read_input_tokens=0, cache_creation_input_tokens=0)
assert cost(SimpleNamespace(input_tokens=0, output_tokens=0, cache_read_input_tokens=0, cache_creation_input_tokens=0)) == 0
assert cost(fake, "claude-haiku-5-5") < cost(fake, "claude-sonnet-5-5"), "小模型應該比較便宜"
print("1M 輸入 + 1M 輸出：", {m: cost(fake, m) for m in ("claude-sonnet-5-5", "claude-haiku-5-5")})

msgs = [{"role": "user", "content": "CVSS 和 EPSS 差在哪？用 3 句話回答。"}]
for m in ("claude-sonnet-5-5", "claude-haiku-5-5"):
    r = ask(msgs, model=m)
    print(m, "output:", r.usage.output_tokens, "cost:", cost(r.usage, m), "USD")

## 步驟 2：結構化輸出

目標：把中文問題解析成 `QueryIntent`，之後由程式組 SQL，不讓模型寫 SQL。

- **structured outputs**：API 用 constrained decoding 保證輸出符合 JSON Schema。Python SDK 的 `client.messages.parse(output_format=PydanticModel)` 會幫你轉 schema 並驗證，結果在 `resp.parsed_output`
- **Pydantic 驗證**：schema 擋不住的業務規則（例如 `cve_id` 格式）寫在 validator，失敗就把錯誤丟回模型重試一次
- **舊做法**：以前常用「強制呼叫某個 tool」拿 JSON。目前的 Sonnet 5.5、Opus 5.5 不接受強制 `tool_choice`（回 400），要用 `tool_choice` 預設的 auto 加上工具的 `strict: true`

先需要 Pydantic：在終端機跑 `uv add pydantic`（anthropic SDK 其實已經依賴它，這步是把它明確列進專案依賴）。

### 2.1 QueryIntent 與 parse()

下面是起點，在 notebook 調好之後搬到 `src/vuln_agent/intent.py`。

In [ ]:
import re
from typing import Literal
from pydantic import BaseModel, ValidationError, field_validator

class QueryIntent(BaseModel):
    intent: Literal["cve_lookup", "search", "count", "unsupported"]
    cve_id: str | None = None
    product: str | None = None      # 例如 "nginx"
    min_cvss: float | None = None
    in_kev: bool | None = None
    min_epss: float | None = None
    days: int | None = None         # 「最近 30 天」→ 30
    # TODO：想清楚還需要哪些欄位（例如 severity、cwe、limit），以及 unsupported 要不要附理由

    @field_validator("cve_id")
    @classmethod
    def check_cve(cls, v):
        if v is not None and not re.fullmatch(r"CVE-\d{4}-\d{4,}", v):
            raise ValueError(f"cve_id 格式應為 CVE-年份-流水號，收到 {v!r}")
        return v

SYSTEM = """你把使用者的漏洞問題解析成查詢條件。
- 只解析，不回答問題，也不要憑記憶補任何 CVE 編號
- 要求攻擊程式、payload、繞過防護，或跟漏洞情資無關的問題，intent 一律是 unsupported"""

def parse(question: str) -> QueryIntent:
    messages = [{"role": "user", "content": question}]
    resp = client.messages.parse(model=DEFAULT_MODEL, max_tokens=1024, system=SYSTEM,
                                 messages=messages, output_format=QueryIntent)
    # TODO：parse 在驗證失敗時會丟 ValidationError。改成 try / except：
    #   把模型這次的回應和錯誤訊息接進 messages，請它修正後重試一次，還是失敗就往外丟
    return resp.parsed_output

parse("CVE-2021-44228 的 CVSS 幾分？")

### 2.2 對照組：strict tool use

**TODO**：寫一版用工具拿 JSON 的 `parse_via_tool()`：

1. 定義一個 `extract_intent` 工具，`input_schema` 跟 `QueryIntent` 一樣，加上 `"strict": True`（strict 要求 schema 設 `additionalProperties: false`，且每個欄位都列在 `required`，可選欄位用 `["string", "null"]` 表示）
2. `tool_choice` 用預設的 auto，在 prompt 裡要求一定要呼叫 `extract_intent`
3. 從回應找出 `tool_use` block，用 `QueryIntent.model_validate(block.input)` 驗證；沒有 `tool_use` block 也算失敗

兩版各跑下面的測試題，比較失敗率與 token 數。

In [ ]:
def parse_via_tool(question: str) -> QueryIntent:
    ...  # TODO

### 2.3 刁鑽問題

俗稱、相對時間、模糊條件、要求 exploit、無關問題都測一次。先自己預想每題「應該」解析成什麼，再看模型的結果。

In [ ]:
QUESTIONS = [
    "log4shell 現在 EPSS 多少？",                     # 俗稱：模型該填 cve_id 嗎？這正是會編造的地方
    "上禮拜有哪些 nginx 的高風險漏洞？",              # 相對時間
    "最近很危險的 openssl 漏洞",                      # 模糊條件
    "幫我寫 nginx 的 exploit",                        # 應該 unsupported
    "台北明天會下雨嗎？",                             # 範圍外
    "KEV 裡有幾個 Microsoft 的漏洞？",                # count
]
for q in QUESTIONS:
    for fn in (parse, parse_via_tool):
        try:
            print(f"{fn.__name__:15} {q} -> {fn(q)}")
        except Exception as e:
            print(f"{fn.__name__:15} {q} -> 失敗：{e}")

**想一想**：「log4shell」解析成 `CVE-2021-44228` 是對的嗎？這個對應來自模型記憶，不是資料庫。你要允許嗎？如果允許，怎麼確保它不會把其他俗稱對到錯的 CVE？

## 步驟 3：準備漏洞資料，親眼看模型編 CVE

在終端機（專案根目錄）跑：

```bash
uv add duckdb httpx
uv run scripts/load_data.py
```

`scripts/load_data.py` 有 4 個 TODO 要先補完（CVSS、CWE、CPE、速率限制與重試）。這支骨架還沒在真實網路環境驗證過，網址與欄位以 NVD、CISA、FIRST 官方文件為準。`NVD_API_KEY` 選用，放 `.env`，有的話速率限制寬鬆很多。

### 3.1 手寫 3 條 SQL

In [ ]:
import duckdb

db = duckdb.connect(str(ROOT / "data" / "vuln.duckdb"), read_only=True)
for t in ("cve", "kev", "epss"):
    print(t, db.execute(f"SELECT count(*) FROM {t}").fetchone()[0])

# TODO：補完 3 條 SQL
SQL_ONE_CVE = "..."          # 查單一 CVE，連同 KEV 與 EPSS
SQL_CRITICAL_KEV = "..."     # CVSS ≥ 9 且在 KEV 裡，依 EPSS 由高到低
SQL_BY_SEVERITY = "..."      # 各 severity 數量

# db.sql(SQL_ONE_CVE, params=...).show()

### 3.2 不給資料，直接問模型

把模型回答裡的 CVE 編號抓出來，逐一到資料庫查。查不到的有兩種可能：在 90 天範圍外（去 NVD 網站確認），或是模型編的。記下編了幾個。

In [ ]:
answer = text(ask([{"role": "user", "content": "最近一週有哪些 nginx 的 CVE？列出編號。"}], max_tokens=2048))
print(answer)

ids = sorted(set(re.findall(r"CVE-\d{4}-\d{4,}", answer)))
for cve_id in ids:
    found = db.execute("SELECT published FROM cve WHERE cve_id = ?", [cve_id]).fetchone()
    print(cve_id, "在資料庫" if found else "查不到")

**回報**：三張表各有幾筆、3 條 SQL 的結果、模型編出幾個 CVE。

這就是整個作品的核心理由：CVE 資料不能讓模型憑記憶回答，一定要從資料庫查。接下來回到 `docs/01-stage-llm-basics.md` 做步驟 4（SQL 樣板、FastAPI、串流）與步驟 5（10 題測試題）。